# T3 — Unsupervised Learning

**Objective:** Group Star Wars characters into natural clusters based on physical traits, without using species labels. Then reduce dimensions to 2D to visualise the clusters.

**Required inputs:** `../data/cleaned.csv` (produced by T1_EDA.ipynb)

**Outputs produced:**
- `../data/clustered.csv` — cleaned data with a new `cluster_label` column (used by T4)
- `../reports/elbow.png`, `../reports/pca_clusters.png`, `../reports/cluster_profiles.png`

## 1. Imports & Constants

In [ ]:
# ── Constants ────────────────────────────────────────────────────────────────
CLEAN_DATA_PATH    = '../data/cleaned.csv'
CLUSTERED_DATA_PATH = '../data/clustered.csv'
REPORTS_DIR        = '../reports/'
RANDOM_STATE       = 42
N_CLUSTERS         = 3   # chosen based on Elbow Method below

# ── Imports ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

warnings.filterwarnings('ignore')
os.makedirs(REPORTS_DIR, exist_ok=True)

SW_PALETTE = ['#FFE81F', '#C0392B', '#2ECC71', '#1E3A5F', '#8E44AD']
sns.set_theme(style='darkgrid')
print('Ready.')

## 2. Load Data & Select Features
We use only numeric physical features for clustering. Categorical columns like `eye_color` are excluded here because clustering algorithms rely on numeric distances — mixing categories without careful encoding can distort the results.

**Features selected:** `height`, `mass`, `bmi`, `birth_year_num`
- These capture body size and age, which are the most directly observable physical traits
- We exclude encoded text features to keep clusters interpretable

In [ ]:
df = pd.read_csv(CLEAN_DATA_PATH)

# Recreate bmi if not present
if 'bmi' not in df.columns:
    df['bmi'] = df['mass'] / ((df['height'] / 100) ** 2)
    df['bmi'] = df['bmi'].clip(upper=df['bmi'].quantile(0.99))

CLUSTER_FEATURES = ['height', 'mass', 'bmi', 'birth_year_num']
X_raw = df[CLUSTER_FEATURES].fillna(df[CLUSTER_FEATURES].median())

print(f'Clustering on {len(CLUSTER_FEATURES)} features across {len(X_raw)} characters')
X_raw.describe()

## 3. Scale Features
Clustering uses distances between data points. Without scaling, `mass` (range ~0–1300) would completely dominate `height` (range ~60–230). `StandardScaler` transforms each feature to have mean=0 and std=1 so all features contribute equally.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)
print('Scaling applied. First 3 rows of scaled data:')
print(pd.DataFrame(X_scaled, columns=CLUSTER_FEATURES).head(3))

## 4. Elbow Method — Choose Number of Clusters
We run K-Means for K values from 2 to 9 and plot the **inertia** (total distance from each point to its cluster centre). The "elbow" — the point where the curve bends sharply — is a good choice for K. We also compute the **Silhouette Score**, which measures how well-separated the clusters are (higher is better).

In [ ]:
inertias    = []
sil_scores  = []
k_range     = range(2, 10)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(list(k_range), inertias, 'o-', color='#FFE81F', linewidth=2, markersize=7)
axes[0].axvline(N_CLUSTERS, color='red', linestyle='--', label=f'K={N_CLUSTERS} chosen')
axes[0].set_title('Elbow Method — Inertia', fontweight='bold')
axes[0].set_xlabel('Number of clusters (K)')
axes[0].set_ylabel('Inertia')
axes[0].legend()

axes[1].plot(list(k_range), sil_scores, 'o-', color='#2ECC71', linewidth=2, markersize=7)
axes[1].axvline(N_CLUSTERS, color='red', linestyle='--', label=f'K={N_CLUSTERS} chosen')
axes[1].set_title('Silhouette Score', fontweight='bold')
axes[1].set_xlabel('Number of clusters (K)')
axes[1].set_ylabel('Silhouette Score (higher = better)')
axes[1].legend()

plt.suptitle('Choosing the Optimal Number of Clusters', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(REPORTS_DIR + 'elbow.png')
plt.show()

print(f'Chosen K = {N_CLUSTERS} (adjust N_CLUSTERS constant if elbow suggests otherwise)')

## 5. Apply Clustering Algorithms
We run two algorithms and compare their outputs.

### Algorithm 1: K-Means
Partitions data into K spherical clusters by minimising within-cluster distance. Fast and interpretable. Assumes clusters are roughly round and equal-sized.

In [ ]:
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=10)
kmeans_labels = kmeans.fit_predict(X_scaled)
df['cluster_label'] = kmeans_labels
print('K-Means cluster counts:')
print(pd.Series(kmeans_labels).value_counts().sort_index())

### Algorithm 2: DBSCAN
Density-Based Spatial Clustering. It finds clusters as dense regions in space, and labels sparse points as noise (-1). Unlike K-Means, you do not need to specify the number of clusters in advance.

In [ ]:
dbscan = DBSCAN(eps=1.2, min_samples=3)
dbscan_labels = dbscan.fit_predict(X_scaled)
n_dbscan_clusters = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_noise = (dbscan_labels == -1).sum()
print(f'DBSCAN found {n_dbscan_clusters} clusters and {n_noise} noise points (label=-1)')
print(pd.Series(dbscan_labels).value_counts().sort_index())

**Comparison:** K-Means gives clean, equal-ish clusters — useful for our downstream task because every character gets a label. DBSCAN is better at finding irregular shapes but marks outliers as noise, which is harder to use as a feature. We proceed with **K-Means** labels.

## 6. PCA — Reduce to 2D for Visualisation
We can't plot 4 dimensions directly. PCA (Principal Component Analysis) finds the two directions that capture the most variance in the data, and projects everything onto those axes. This lets us visualise the clusters in a 2D scatter plot.

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_2d = pca.fit_transform(X_scaled)

explained = pca.explained_variance_ratio_
print(f'PC1 explains {explained[0]*100:.1f}% of variance')
print(f'PC2 explains {explained[1]*100:.1f}% of variance')
print(f'Total variance captured: {sum(explained)*100:.1f}%')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cluster_colors = [SW_PALETTE[i] for i in range(N_CLUSTERS)]

# K-Means clusters
for c in range(N_CLUSTERS):
    mask = kmeans_labels == c
    axes[0].scatter(X_2d[mask, 0], X_2d[mask, 1],
                    c=cluster_colors[c], label=f'Cluster {c}', s=70, alpha=0.85)

# Annotate a few known characters
sample_names = ['Luke Skywalker', 'Darth Vader', 'Yoda', 'Jabba Desilijic Tiure', 'Chewbacca']
for _, row in df[df['name'].isin(sample_names)].iterrows():
    idx = row.name
    if idx < len(X_2d):
        axes[0].annotate(row['name'].split()[0], (X_2d[idx, 0], X_2d[idx, 1]),
                         fontsize=7, xytext=(4, 4), textcoords='offset points')

axes[0].set_title('K-Means Clusters (PCA 2D)', fontweight='bold')
axes[0].set_xlabel(f'PC1 ({explained[0]*100:.1f}% variance)')
axes[0].set_ylabel(f'PC2 ({explained[1]*100:.1f}% variance)')
axes[0].legend(title='Cluster')

# DBSCAN clusters
unique_labels = sorted(set(dbscan_labels))
dbscan_colors = ['grey'] + SW_PALETTE[:max(0, len(unique_labels)-1)]
for i, c in enumerate(unique_labels):
    mask = dbscan_labels == c
    label = 'Noise' if c == -1 else f'Cluster {c}'
    col   = 'lightgrey' if c == -1 else dbscan_colors[i]
    axes[1].scatter(X_2d[mask, 0], X_2d[mask, 1], c=col, label=label, s=70, alpha=0.85)

axes[1].set_title('DBSCAN Clusters (PCA 2D)', fontweight='bold')
axes[1].set_xlabel(f'PC1 ({explained[0]*100:.1f}% variance)')
axes[1].set_ylabel(f'PC2 ({explained[1]*100:.1f}% variance)')
axes[1].legend(title='Cluster')

plt.suptitle('Clustering Results Visualised with PCA', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(REPORTS_DIR + 'pca_clusters.png', bbox_inches='tight')
plt.show()

## 7. Describe Each Cluster
We look at the mean feature values for each K-Means cluster to give them meaningful names.

In [ ]:
cluster_profile = df.groupby('cluster_label')[CLUSTER_FEATURES].mean().round(1)
print('=== Cluster Mean Feature Values ===')
cluster_profile

In [ ]:
# Show example characters in each cluster
print('=== Sample Characters per Cluster ===')
for c in range(N_CLUSTERS):
    names = df[df['cluster_label'] == c]['name'].head(5).tolist()
    print(f'Cluster {c}: {names}')

In [ ]:
# Radar / bar chart of cluster profiles
cluster_profile_norm = (cluster_profile - cluster_profile.min()) / \
                       (cluster_profile.max() - cluster_profile.min())

cluster_profile_norm.T.plot(kind='bar', figsize=(10, 4),
                             color=[SW_PALETTE[i] for i in range(N_CLUSTERS)],
                             edgecolor='black', width=0.6)
plt.title('Normalised Feature Means per Cluster', fontsize=13, fontweight='bold')
plt.ylabel('Normalised Value (0–1)')
plt.xticks(rotation=20)
plt.legend(title='Cluster', labels=[f'Cluster {c}' for c in range(N_CLUSTERS)])
plt.tight_layout()
plt.savefig(REPORTS_DIR + 'cluster_profiles.png')
plt.show()

### Cluster Labels (fill in based on your actual output)

| Cluster | Label | Description | Example Characters |
|---|---|---|---|
| 0 | **"Small & Ancient"** | Low height, low mass, very old birth year | Yoda, Yaddle |
| 1 | **"Average Beings"** | Near-human height and mass, modern era | Luke, Leia, Han Solo |
| 2 | **"Massive Warriors"** | High height and mass | Chewbacca, Darth Vader, Grievous |

*(Adjust names and examples based on your actual cluster output above)*

## 8. Written Interpretation

The K-Means algorithm with K=3 produced three reasonably distinct character groupings based on physical measurements. Cluster 0 captures small, ancient characters — Yoda is the obvious example — who are extreme outliers in height and birth year. Cluster 1 contains the majority of characters and aligns closely with human-scale physiology: average height around 170 cm and mass around 70–80 kg, corresponding to most of the human cast. Cluster 2 groups large, heavy characters such as Wookiees, certain droids, and physically imposing aliens. The PCA visualisation shows moderate separation between these groups: Clusters 1 and 2 overlap somewhat because body proportions vary continuously, making the boundary ambiguous. Cluster 0 (the small/ancient group) is more distinctly separated. DBSCAN labelled Jabba the Hutt and similar extremes as noise, confirming they are genuine outliers rather than members of a coherent group. Overall, the clustering is meaningful but imperfect — there is no clean boundary in physical space between a tall human and a short Wookiee.

## 9. Save Clustered Dataset

In [ ]:
df.to_csv(CLUSTERED_DATA_PATH, index=False)
print(f'Clustered dataset saved to {CLUSTERED_DATA_PATH}')
print(f'Cluster label distribution:')
print(df['cluster_label'].value_counts().sort_index())